# import

In [13]:
import os

import pandas as pd
import pandas.io.sql as sqlio
import psycopg2 as ps


# 2. Análise de negócios
## Ligação com bancos de dados
Fomos contratados para formular uma análise dos preços históricos de combustíveis e retornar com uma estratégia de negócios.

Os dados para esse exercício serão retirados da [ANP](https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/serie-historica-de-precos-de-combustiveis).

Utilizaremos o Postgre17 via pgadmin.

### PostgresSQL
Criamos uma database nova chamada 'ANP' no nossos servidor PostreSQL 17.
Encoding UTF-8.

Em `Schemas` guardaremos as informações sobre nossos dados. (owner: `postgres`).

As queries estão salvas em `/queries`.

### Extract, Transform and Load (ETL - KNIME)
Faremos nossa injeção de dados pelo `KNIME`. Vamos converter todos nossos `.csv`s para `.xlsx` para facilidade de leitura.

In [4]:
def to_float(df, column):
    df[column] = df[column].replace(',', '.', regex=True)
    df[column] = df[column].astype(float)
    return df

In [5]:
def to_date(df, column):
    df[column] = pd.to_datetime(df[column], format = '%d/%m/%Y')
    return df

In [12]:
# df = pd.read_csv('local/abt/ca-2019-01.csv', sep = ';', encoding = 'ISO-8859-1')
# df = pd.read_excel('local/abt/xlsx/ca-2019-01.xlsx')

In [7]:
def convert_csv_xlsx(
    directory,
    save_to = '',
    sep : str = ';',
    encoding = 'ISO-8859-1'
    ):
    # directory = 'local/'
    files = os.listdir(directory)
    for f in files:
        path = os.path.join(directory, f)
        is_file = f.endswith('csv') and os.path.isfile(path)
        if is_file:
            try:
                print(f'\nAttempting to convert {f}')
                file_name = f.split('.')[0]
                temp = pd.read_csv(path, sep = sep, encoding = encoding)

                temp = to_float(temp, 'Valor de Venda')
                temp = to_date(temp, 'Data da Coleta')

                temp.to_excel(os.path.join(save_to, f'{file_name}.xlsx'))
            except Exception as e:
                print(f'Failed {f}')
                print(e)

In [8]:
convert_csv_xlsx('local/abt/', save_to = 'local/abt/xlsx/')


Attempting to convert ca-2019-01.csv

Attempting to convert ca-2019-02.csv

Attempting to convert ca-2020-01.csv

Attempting to convert ca-2020-02.csv


D:\Temp\ipykernel_38228\553607199.py:16: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(path, sep = sep, encoding = encoding)



Attempting to convert ca-2021-01.csv

Attempting to convert ca-2021-02.csv

Attempting to convert ca-2022-01.csv

Attempting to convert ca-2022-02.csv

Attempting to convert ca-2023-01.csv

Attempting to convert ca-2023-02.csv

Attempting to convert ca-2024-01.csv

Attempting to convert ca-2024-02.csv


Conectamos o KNIME a nossa base de dados utilizando os dados em `PostgreSQL 17` > `Properties`, etc

### Conectando ao banco de dados ao VS CODE

In [16]:
dbname = 'ANP'
user = 'postgres'
password = ''
with open('./local/postgres-pw') as f:
    password = f.read()
host = 'localhost'
port = '5432'


conn = ps.connect(
    dbname = dbname,
    user = user,
    password = password,
    host = host,
    port = port
)

In [20]:
sql = '''
SELECT * FROM anp.preco_combustivel LIMIT 5
'''

In [21]:
df = sqlio.read_sql_query(sql, conn)
df

D:\Temp\ipykernel_38228\4176714469.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = sqlio.read_sql_query(sql, conn)


,regiao,estado,municipio,revenda,cnpj,nome_rua,numero_rua,complemento,bairro,cep,produto,data_coleta,valor_venda,unidade_medida,bandeira
0,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,GASOLINA,2019-01-03,4.199,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
1,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,ETANOL,2019-01-03,2.899,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
2,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,DIESEL S10,2019-01-03,3.349,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
3,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,GNV,2019-01-03,2.439,R$ / mÂ³,PETROBRAS DISTRIBUIDORA S.A.
4,S,RS,CANOAS,METROPOLITANO COMERCIO DE COMBUSTIVEIS LTDA,88.587.589/0001-17,AVENIDA GUILHERME SCHELL,6340,None,CENTRO,92310-000,GASOLINA,2019-01-02,4.399,R$ / litro,BRANCA
